# El problema de las N reinas con algoritmos genéticos
**Fundamentos de Inteligencia Artificial** · Basado en *Algoritmos Evolutivos: un enfoque práctico* (Araujo y Cervigón), sección **3.5.2.5 "El problema de las N reinas"** (págs. 77–79), con los operadores para permutaciones de la sección **4.1.1** (págs. 94–105) y los componentes de los capítulos 2 y 3.

**Alumno:** Myranda Belen Infante Castillo · **Matrícula:** 2430384 · ITIID · UPV

### El problema
Colocar $N$ reinas en un tablero de ajedrez de $N\times N$ de manera que ninguna esté amenazada: no puede haber dos reinas en la misma fila, en la misma columna ni en la misma diagonal.

Se buscan $N$ variables $\{R_1,\dots,R_N\}$ con valores en $\{1..N\}$, donde $R_i$ es la fila de la reina de la columna $i$, que cumplan:

$$R_i \neq R_j \qquad\qquad |R_i - R_j| \neq |i - j|$$

La primera restricción impide dos reinas en la misma fila; la segunda, dos reinas en la misma diagonal (principal o secundaria). La de las columnas se cumple sola, porque cada columna tiene exactamente una reina.

### Diseño del algoritmo (según el libro)

| Elemento | Opción del libro | Alternativas incluidas |
|---|---|---|
| Representación | Vector de $N$ enteros: posición $i$ = columna, valor = fila | — |
| Población inicial | Permutaciones de $1..N$ (cumplen $R_i \neq R_j$) | Enteros aleatorios (sin restricción) |
| Adaptación | **Crawford**: número de conflictos (minimizar, penalización) | **Homaifar**: premio de $\frac{1}{2N}$ por diagonal libre (maximizar) |
| Selección | Ruleta (cap. 2) | Torneo |
| Cruce | Monopunto o dos puntos, con porcentaje reducido | PMX, OX y por ciclos (cap. 4, conservan la permutación) |
| Mutación | Intercambio de dos posiciones | Inserción, inversión, heurística (cap. 4) y aleatoria |
| Mejoras | Elitismo (3.2) y parada al encontrar una solución (3.3) | — |

### Cómo usar este cuaderno
1. *Entorno de ejecución → Ejecutar todas* (Ctrl+F9) la primera vez.
2. Para cambiar $N$, la función de adaptación, el cruce, la mutación o cualquier parámetro, usa el **formulario de la celda 3** y vuelve a *Ejecutar todas*.
3. La sección 8 da los resultados y las tablas; la 10, las gráficas de cada generación; la 11 compara con un algoritmo determinista; la 12, el estudio de parámetros.

In [ ]:
#@title 1. Librerías
import math
import time
import random
import itertools
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from matplotlib.animation import FuncAnimation, PillowWriter
from IPython.display import HTML, display

plt.rcParams["animation.embed_limit"] = 80   # MB, para animaciones largas en Colab
plt.rcParams["figure.dpi"] = 100
pd.set_option("display.max_rows", 300)
print("Librerías cargadas correctamente")

## 2. El "ambiente": restricciones y funciones de adaptación
El libro presenta dos funciones de adaptación:

**Crawford [CRA92]** — *penalización, minimización.* Para cada reina $q_i$ se comprueba con las restantes $q_j$ si $\lvert q_i - q_j\rvert = \lvert i - j\rvert$, y se suma uno por cada conflicto. Aquí se cuenta cada **pareja** de reinas que se ataca una sola vez (contar ambos sentidos solo duplica el valor y no cambia el óptimo). Cuando la representación permite reinas en la misma fila, esas parejas también cuentan como conflicto.
$$f_{Crawford} = \#\{(i,j),\ i<j : q_i = q_j \ \text{ o } \ \lvert q_i-q_j\rvert = \lvert i-j\rvert\}, \qquad \text{óptimo} = 0$$

**Homaifar [HOM92]** — *premio, maximización.* Por cada reina y por cada una de sus dos diagonales, si en esa diagonal no hay otra reina se suma $\frac{1}{2N}$:
$$f_{Homaifar} = \sum_{i=1}^{N}\ \sum_{d\,\in\,\{\searrow,\ \nearrow\}} \frac{1}{2N}\,[\text{la diagonal } d \text{ de la reina } i \text{ está libre}], \qquad \text{óptimo} = 1$$

> **Nota para el docente.** La fórmula de Homaifar solo revisa diagonales porque supone que los individuos son permutaciones (nunca hay dos reinas en la misma fila). Si se eligen operadores que pueden repetir filas (cruce monopunto o de dos puntos, inicialización aleatoria o mutación aleatoria), el cuaderno también revisa la fila de cada reina y el premio pasa a $\frac{1}{3N}$ por línea libre, de modo que el máximo sigue siendo 1 y solo lo alcanzan las soluciones reales.

Abajo se dibuja el ejemplo del libro para $N=4$: el individuo **[2, 4, 1, 3]**.

In [ ]:
#@title 2. Restricciones, funciones de adaptación y dibujo del tablero
def contar_conflictos(genes):
    """Número de PAREJAS de reinas que se atacan (misma fila o misma diagonal)."""
    filas, diag, anti = {}, {}, {}
    for col, fila in enumerate(genes, start=1):
        filas[fila] = filas.get(fila, 0) + 1
        diag[fila - col] = diag.get(fila - col, 0) + 1     # diagonal principal  (\)
        anti[fila + col] = anti.get(fila + col, 0) + 1     # diagonal secundaria (/)
    return sum(c * (c - 1) // 2 for d in (filas, diag, anti) for c in d.values())

def parejas_en_conflicto(genes):
    """Lista de parejas (i, j) de columnas cuyas reinas se atacan."""
    N = len(genes)
    return [(i, j) for i in range(N) for j in range(i + 1, N)
            if genes[i] == genes[j] or abs(genes[i] - genes[j]) == abs(i - j)]

def adaptacion_crawford(genes):
    return contar_conflictos(genes)

def adaptacion_homaifar(genes, revisar_filas=False):
    N = len(genes)
    filas, diag, anti = {}, {}, {}
    for col, fila in enumerate(genes, start=1):
        filas[fila] = filas.get(fila, 0) + 1
        diag[fila - col] = diag.get(fila - col, 0) + 1
        anti[fila + col] = anti.get(fila + col, 0) + 1
    lineas = 3 if revisar_filas else 2
    premio = 1.0 / (lineas * N)
    f = 0.0
    for col, fila in enumerate(genes, start=1):
        if diag[fila - col] == 1:
            f += premio
        if anti[fila + col] == 1:
            f += premio
        if revisar_filas and filas[fila] == 1:
            f += premio
    return f

# Cruces que garantizan que los hijos siguen siendo permutaciones (sección 4.1.1.2)
CRUCES_PERMUTACION = {"PMX", "OX", "ciclos"}

# Número exacto de soluciones conocidas (para comparar)
SOLUCIONES_CONOCIDAS = {1: 1, 2: 0, 3: 0, 4: 2, 5: 10, 6: 4, 7: 40, 8: 92, 9: 352, 10: 724,
                        11: 2680, 12: 14200, 13: 73712, 14: 365596, 15: 2279184}

# ---------- Dibujo del tablero ----------
CASILLA_CLARA  = "#F0E6D2"
CASILLA_OSCURA = "#B9A88C"
COLOR_ATAQUE   = "#D55E00"   # reinas atacadas y líneas de ataque
COLOR_REINA    = "#1A1A1A"   # reinas a salvo

def dibujar_tablero(ax, genes, titulo=None, mostrar_ataques=True, mostrar_vector=True):
    N = len(genes)
    tablero = np.indices((N, N)).sum(axis=0) % 2
    ax.imshow(tablero, cmap=ListedColormap([CASILLA_CLARA, CASILLA_OSCURA]),
              extent=(0.5, N + 0.5, N + 0.5, 0.5), zorder=0)
    ataques = parejas_en_conflicto(genes)
    atacadas = {k for par in ataques for k in par}
    if mostrar_ataques:
        for i, j in ataques:
            ax.plot([i + 1, j + 1], [genes[i], genes[j]], color=COLOR_ATAQUE,
                    lw=1.4, alpha=0.65, zorder=1)
    tam = max(5, min(30, 230 / N))
    for col, fila in enumerate(genes, start=1):
        ax.text(col, fila, "♛", ha="center", va="center", fontsize=tam, zorder=2,
                color=COLOR_ATAQUE if (col - 1) in atacadas else COLOR_REINA)
    ax.set_xlim(0.5, N + 0.5)
    ax.set_ylim(N + 0.5, 0.5)
    paso = 1 if N <= 12 else max(1, N // 8)
    ax.set_xticks(range(1, N + 1, paso))
    ax.set_yticks(range(1, N + 1, paso))
    ax.xaxis.tick_top()
    ax.tick_params(length=0, labelsize=8)
    for s in ax.spines.values():
        s.set_visible(False)
    if titulo is None:
        conf = contar_conflictos(genes)
        estado = "SOLUCIÓN" if conf == 0 else f"{conf} conflicto(s)"
        vector = ("[" + ", ".join(map(str, genes)) + "]\n") if (mostrar_vector and N <= 16) else ""
        titulo = vector + estado
    ax.set_title(titulo, fontsize=9, pad=14 if N <= 12 else 8)

# Ejemplo del libro (N = 4) y un individuo que NO es solución
fig, axes = plt.subplots(1, 2, figsize=(7, 3.8))
dibujar_tablero(axes[0], [2, 4, 1, 3], titulo="Ejemplo del libro: [2, 4, 1, 3]\nSOLUCIÓN (0 conflictos)")
dibujar_tablero(axes[1], [1, 2, 3, 4])
plt.tight_layout()
plt.show()

for g in ([2, 4, 1, 3], [1, 2, 3, 4], [1, 1, 1, 1]):
    print(f"{g}:  Crawford = {adaptacion_crawford(g)} conflictos  |  "
          f"Homaifar = {adaptacion_homaifar(g):.4f}  |  Homaifar revisando filas = {adaptacion_homaifar(g, True):.4f}")
print("\nObserva [1, 1, 1, 1]: las cuatro reinas están en la misma fila, pero la fórmula original de Homaifar"
      "\nle da 1.0 porque solo revisa diagonales. Por eso, si los operadores pueden repetir filas, se revisan también.")

## 3. Parámetros (formulario)
Todo lo que pida el doctor se cambia aquí:
- `N`: tamaño del tablero (número de reinas).
- `ADAPTACION`: `crawford` (minimizar conflictos) o `homaifar` (maximizar el premio).
- `INICIALIZACION`: `permutacion` (como el libro: cumple $R_i \neq R_j$) o `aleatoria`.
- `SELECCION`: `ruleta` (como el libro) o `torneo` (con `TAM_TORNEO` individuos).
- `CRUCE`: `monopunto` y `dos_puntos` (los que menciona la sección 3.5.2.5) o `PMX`, `OX` y `ciclos` (sección 4.1.1.2).
- `PROB_CRUCE`: porcentaje de cruces. El libro recomienda que sea **reducido** con monopunto o dos puntos, porque esos cruces suelen romper las soluciones.
- `MUTACION`: `intercambio` (la del libro), `insercion`, `inversion`, `heuristica` o `aleatoria`.
- `PROB_MUT`: probabilidad de que **cada individuo** sea mutado.
- `DETENER_AL_ENCONTRAR`: termina en cuanto aparece una solución (criterio de la sección 3.3). Si está apagado, el algoritmo sigue y va acumulando **soluciones distintas**.
- `SEMILLA = -1` da resultados distintos cada vez; cualquier otro número hace el experimento repetible.

> PMX, OX y ciclos solo funcionan con permutaciones: requieren `INICIALIZACION = permutacion` y una mutación distinta de `aleatoria`. Si se combinan mal, el cuaderno lo avisa.

In [ ]:
#@title 3. Parámetros del algoritmo genético (N reinas)
N = 8 #@param {type:"integer"}
ADAPTACION = "crawford" #@param ["crawford", "homaifar"]
INICIALIZACION = "permutacion" #@param ["permutacion", "aleatoria"]
TAM_POB = 100 #@param {type:"integer"}
NUM_GENERACIONES = 150 #@param {type:"integer"}
SELECCION = "ruleta" #@param ["ruleta", "torneo"]
TAM_TORNEO = 3 #@param {type:"integer"}
CRUCE = "PMX" #@param ["monopunto", "dos_puntos", "PMX", "OX", "ciclos"]
PROB_CRUCE = 0.6 #@param {type:"slider", min:0, max:1, step:0.05}
MUTACION = "intercambio" #@param ["intercambio", "insercion", "inversion", "heuristica", "aleatoria"]
PROB_MUT = 0.2 #@param {type:"slider", min:0, max:1, step:0.05}
ELITISMO = True #@param {type:"boolean"}
PORC_ELITE = 0.02 #@param {type:"number"}
DETENER_AL_ENCONTRAR = False #@param {type:"boolean"}
SEMILLA = 42 #@param {type:"integer"}

CONFIG = dict(N=N, ADAPTACION=ADAPTACION, INICIALIZACION=INICIALIZACION, TAM_POB=TAM_POB,
              NUM_GENERACIONES=NUM_GENERACIONES, SELECCION=SELECCION, TAM_TORNEO=TAM_TORNEO,
              CRUCE=CRUCE, PROB_CRUCE=PROB_CRUCE, MUTACION=MUTACION, PROB_MUT=PROB_MUT,
              ELITISMO=ELITISMO, PORC_ELITE=PORC_ELITE,
              DETENER_AL_ENCONTRAR=DETENER_AL_ENCONTRAR, SEMILLA=SEMILLA)
CONFIG

## 4. Representación y espacio de búsqueda
Como dos reinas no pueden estar en la misma columna, se asigna **una reina a cada columna** y se identifica cada reina por su fila. El individuo es un vector de $N$ enteros donde el valor de la posición $i$ es la fila de la reina de la columna $i$ (ejemplo del libro: **[2, 4, 1, 3]**).

- Con la inicialización **aleatoria** cada gen puede valer de 1 a $N$: hay $N^N$ individuos posibles.
- Con **permutaciones** la restricción de filas se cumple siempre: el espacio baja a $N!$ individuos. Esta es una **técnica de codificación** (sección 3.5.1), porque la representación misma garantiza parte de las restricciones; la función de Crawford, en cambio, es una **técnica de penalización**.

In [ ]:
#@title 4. Tamaño del espacio de búsqueda
print(f"N = {N}")
print(f"  Individuos posibles con enteros aleatorios (N^N): {N**N:,}")
print(f"  Individuos posibles con permutaciones (N!):      {math.factorial(N):,}")
if N in SOLUCIONES_CONOCIDAS:
    s = SOLUCIONES_CONOCIDAS[N]
    print(f"  Soluciones que existen: {s:,}")
    if s:
        print(f"  Probabilidad de acertar con UNA permutación al azar: 1 en {math.factorial(N) / s:,.0f}")
        print(f"  Probabilidad de acertar con UN vector al azar:      1 en {N**N / s:,.0f}")

## 5. Clase `Individuo` (genotipo → fenotipo → adaptación)
Como se vio en clase:
- `genes` — **genotipo**: el vector de enteros, ej. `[2, 4, 1, 3]`.
- `x` — **fenotipo**: lo que "se ve", las casillas (fila, columna) donde quedan las reinas en el tablero.
- `adapt` — **adaptación**: la salida de la función (Crawford o Homaifar).
- `conflictos` — se calcula siempre, aunque la adaptación sea la de Homaifar, para poder comparar.

Además, como en el libro (2.9.1): `aptitud` (adaptación revisada para la ruleta), `puntuacion` y `punt_acu`.

In [ ]:
#@title 5. Clase Individuo y preparación del problema
def preparar_problema(cfg):
    if int(cfg["N"]) < 4:
        raise ValueError("N debe ser al menos 4 (para N = 2 y N = 3 no existe ninguna solución).")
    perm = (cfg["INICIALIZACION"] == "permutacion" and cfg["MUTACION"] != "aleatoria"
            and cfg["CRUCE"] in CRUCES_PERMUTACION)
    if cfg["CRUCE"] in CRUCES_PERMUTACION and (cfg["INICIALIZACION"] != "permutacion"
                                               or cfg["MUTACION"] == "aleatoria"):
        raise ValueError("Los cruces PMX, OX y por ciclos necesitan individuos que sean permutaciones: "
                         "usa INICIALIZACION = 'permutacion' y una MUTACION distinta de 'aleatoria'.")
    if cfg["ADAPTACION"] == "crawford":
        f, objetivo, optimo = adaptacion_crawford, "min", 0.0
        formula = "número de parejas de reinas que se atacan"
    else:
        revisar_filas = not perm
        f = lambda g: adaptacion_homaifar(g, revisar_filas)
        objetivo, optimo = "max", 1.0
        formula = f"premio 1/({3 if revisar_filas else 2}N) por {'línea' if revisar_filas else 'diagonal'} libre"
    return dict(N=int(cfg["N"]), f=f, objetivo=objetivo, optimo=optimo, formula=formula,
                adaptacion=cfg["ADAPTACION"], inicializacion=cfg["INICIALIZACION"],
                siempre_permutacion=perm, evaluaciones=[0])

class Individuo:
    def __init__(self, problema, genes=None):
        self.problema = problema
        if genes is None:
            n = problema["N"]
            if problema["inicializacion"] == "permutacion":
                genes = random.sample(range(1, n + 1), n)          # permutación de 1..N
            else:
                genes = [random.randint(1, n) for _ in range(n)]    # cada gen de 1 a N
        self.genes = list(genes)  # genotipo
        self.x = None             # fenotipo: casillas (fila, columna) de las reinas
        self.conflictos = None    # parejas de reinas que se atacan
        self.adapt = None         # adaptación f(x)
        self.aptitud = 0.0        # adaptación revisada (positiva) que usa la ruleta
        self.puntuacion = 0.0     # aptitud / suma de aptitudes
        self.punt_acu = 0.0       # puntuación acumulada para los sorteos
        self.evaluar()

    def decodificar(self):
        self.x = [(fila, col) for col, fila in enumerate(self.genes, start=1)]
        return self.x

    def evaluar(self):
        self.decodificar()
        self.conflictos = contar_conflictos(self.genes)
        self.adapt = float(self.problema["f"](self.genes))
        self.problema["evaluaciones"][0] += 1
        return self.adapt

    def es_solucion(self):
        return self.conflictos == 0

    def copia(self):
        nuevo = Individuo.__new__(Individuo)          # copia sin volver a evaluar
        nuevo.__dict__.update(self.__dict__)
        nuevo.genes = self.genes[:]
        nuevo.x = list(self.x)
        return nuevo

    def cadena(self):
        return "[" + ", ".join(map(str, self.genes)) + "]"

    def __repr__(self):
        return f"genes={self.cadena()} | conflictos={self.conflictos} | adapt={self.adapt:.4f}"

# Prueba rápida
PROBLEMA = preparar_problema(CONFIG)
print("Adaptación:", PROBLEMA["adaptacion"], "→", PROBLEMA["formula"],
      f"({'minimizar' if PROBLEMA['objetivo'] == 'min' else 'maximizar'}, óptimo = {PROBLEMA['optimo']})")
random.seed(1)
for _ in range(3):
    print(Individuo(PROBLEMA))

## 6. Operadores comunes (capítulos 2 y 3 del libro)
- **Revisión de la adaptación** (3.1.1): la ruleta necesita valores positivos donde "más grande" sea "mejor".
  - Crawford (minimizar): `aptitud = cmax·1.05 − conflictos`, donde `cmax` es el mayor número de conflictos de la población. El 5 % extra evita que la suma sea 0 cuando todos tienen los mismos conflictos.
  - Homaifar (maximizar): `aptitud = f(x)`, que ya es positiva.
- **Evaluación** (2.9.4): `puntuacion = aptitud / sumadaptacion` y la `punt_acu` acumulada para los sorteos.
- **Selección por ruleta** (2.9.5) o **por torneo**: el torneo toma `TAM_TORNEO` individuos al azar y se queda con el mejor.
- **Reproducción** (2.9.6): cada individuo entra al cruce con probabilidad `PROB_CRUCE`, se cruzan por parejas y **los hijos sustituyen a los padres**.
- **Elitismo** (3.2): se reservan los mejores `E` individuos (1–2 % de la población) y al final de la generación reemplazan a los peores.

In [ ]:
#@title 6. Operadores comunes: adaptación, evaluación, selección, reproducción y elitismo
def es_mejor(a, b, objetivo):
    return a > b if objetivo == "max" else a < b

def revisar_adaptacion(pob, objetivo):
    """Sección 3.1.1: aptitud positiva para la ruleta."""
    if objetivo == "min":
        cmax = max(ind.adapt for ind in pob)
        cmax = cmax + 0.05 * abs(cmax)            # = cmax*1.05 (cmax >= 0 en Crawford)
        for ind in pob:
            ind.aptitud = cmax - ind.adapt
    else:
        for ind in pob:
            ind.aptitud = ind.adapt

def evaluacion(pob, objetivo):
    """Sección 2.9.4: puntuación relativa, puntuación acumulada y posición del mejor."""
    sumadap = sum(ind.aptitud for ind in pob)
    pos_mejor = 0
    for i, ind in enumerate(pob):
        if es_mejor(ind.adapt, pob[pos_mejor].adapt, objetivo):
            pos_mejor = i
    punt_acu = 0.0
    for ind in pob:
        ind.puntuacion = ind.aptitud / sumadap if sumadap > 0 else 1.0 / len(pob)
        punt_acu += ind.puntuacion
        ind.punt_acu = punt_acu
    pob[-1].punt_acu = 1.0
    return pos_mejor, sumadap

def seleccion_ruleta(pob, cfg):
    """Sección 2.9.5: tam_pob supervivientes por el método de la ruleta."""
    nueva = []
    for _ in range(len(pob)):
        prob = random.random()
        pos_super = 0
        while prob > pob[pos_super].punt_acu and pos_super < len(pob) - 1:
            pos_super += 1
        nueva.append(pob[pos_super].copia())
    return nueva

def seleccion_torneo(pob, cfg):
    """Torneo determinista: gana el de mayor aptitud entre TAM_TORNEO elegidos al azar."""
    k = max(2, min(int(cfg["TAM_TORNEO"]), len(pob)))
    return [max(random.sample(pob, k), key=lambda ind: ind.aptitud).copia() for _ in range(len(pob))]

SELECCIONES = {"ruleta": seleccion_ruleta, "torneo": seleccion_torneo}

def reproduccion(pob, prob_cruce, cruce):
    """Sección 2.9.6: los hijos sustituyen a sus progenitores."""
    sel_cruce = [i for i in range(len(pob)) if random.random() < prob_cruce]
    if len(sel_cruce) % 2 == 1:
        sel_cruce.pop()
    for k in range(0, len(sel_cruce), 2):
        a, b = sel_cruce[k], sel_cruce[k + 1]
        pob[a], pob[b] = cruce(pob[a], pob[b])

def obtener_elite(pob, tam_elite, objetivo):
    if tam_elite <= 0:
        return []
    ordenados = sorted(pob, key=lambda ind: ind.adapt, reverse=(objetivo == "max"))
    return [ind.copia() for ind in ordenados[:tam_elite]]

def incluir_elite(pob, elite, objetivo):
    peores = sorted(range(len(pob)), key=lambda i: pob[i].adapt, reverse=(objetivo == "min"))
    for pos, ind in zip(peores, elite):
        pob[pos] = ind

def registrar(pob, generacion, pos_mejor):
    """Guarda una "foto" de la generación para las tablas y las gráficas."""
    G = np.array([ind.genes for ind in pob])
    F = np.array([ind.adapt for ind in pob])
    Cf = np.array([ind.conflictos for ind in pob])
    soluciones = {tuple(ind.genes) for ind in pob if ind.conflictos == 0}
    return dict(generacion=generacion, G=G, F=F, C=Cf,
                mejor_genes=list(pob[pos_mejor].genes), mejor_f=float(pob[pos_mejor].adapt),
                mejor_conf=int(pob[pos_mejor].conflictos), media=float(F.mean()),
                media_conf=float(Cf.mean()), soluciones=soluciones,
                diversidad=len({tuple(g) for g in G}),
                evaluaciones=pob[0].problema["evaluaciones"][0])

## 7. Operadores de cruce y mutación, y el algoritmo genético
**Cruces de la sección 3.5.2.5** (sencillos, pero *"el simple intercambio de partes de dos soluciones en general da lugar a individuos que no son solución"*, por eso su porcentaje debe ser reducido):
- **Monopunto:** un punto al azar; se intercambian las colas.
- **Dos puntos:** se intercambia el segmento entre dos puntos al azar.

**Cruces para permutaciones de la sección 4.1.1.2** (los hijos siempre son permutaciones, así que nunca hay dos reinas en la misma fila):
- **PMX** (emparejamiento parcial): se intercambian las subcadenas entre dos cortes; los valores repetidos se resuelven con los emparejamientos de esas subcadenas.
- **OX** (por orden): cada hijo copia la subcadena de un padre y completa con los valores del otro, empezando después del segundo corte y respetando su orden.
- **Por ciclos:** se busca el ciclo de posiciones que empieza en el primer elemento; esas posiciones se heredan de un padre y el resto del otro.

**Mutaciones** (se aplican a cada individuo con probabilidad `PROB_MUT`):
- **Intercambio** (la que propone la sección 3.5.2.5): se intercambian los valores de dos posiciones.
- **Inserción**: un valor se saca de su posición y se inserta en otra, desplazando el resto.
- **Inversión**: se invierte el tramo entre dos puntos.
- **Heurística**: se eligen 3 posiciones, se prueban todas sus permutaciones y se queda la mejor.
- **Aleatoria**: un gen cambia a una fila al azar (rompe las permutaciones).

Bucle (2.9): selección → reproducción → mutación → elitismo → revisión de la adaptación → evaluación, hasta `NUM_GENERACIONES` o hasta encontrar una solución (si `DETENER_AL_ENCONTRAR` está activado).

In [ ]:
#@title 7. Cruces, mutaciones y algoritmo genético
def _dos_cortes(n):
    a, b = sorted(random.sample(range(n + 1), 2))
    return a, b

# ---------------- CRUCES (sobre vectores de genes) ----------------
def cruce_monopunto(p1, p2, punto=None):
    n = len(p1)
    k = punto if punto is not None else random.randint(1, n - 1)
    return p1[:k] + p2[k:], p2[:k] + p1[k:]

def cruce_dos_puntos(p1, p2, cortes=None):
    a, b = cortes if cortes is not None else _dos_cortes(len(p1))
    return p1[:a] + p2[a:b] + p1[b:], p2[:a] + p1[a:b] + p2[b:]

def cruce_pmx(p1, p2, cortes=None):
    n = len(p1)
    a, b = cortes if cortes is not None else _dos_cortes(n)
    def hijo(base, otro):                 # subcadena de 'otro', resto de 'base'
        h = base[:]
        h[a:b] = otro[a:b]
        pareja = {otro[i]: base[i] for i in range(a, b)}
        for i in list(range(a)) + list(range(b, n)):
            v, pasos = base[i], 0
            while v in pareja and pasos < n:   # se sigue la cadena de emparejamientos
                v, pasos = pareja[v], pasos + 1
            h[i] = v
        return h
    return hijo(p1, p2), hijo(p2, p1)

def cruce_ox(p1, p2, cortes=None):
    n = len(p1)
    a, b = cortes if cortes is not None else _dos_cortes(n)
    def hijo(fuente, otro):               # subcadena de 'fuente', orden de 'otro'
        h = [None] * n
        h[a:b] = fuente[a:b]
        usados = set(h[a:b])
        resto = [otro[(b + k) % n] for k in range(n) if otro[(b + k) % n] not in usados]
        huecos = [(b + k) % n for k in range(n - (b - a))]
        for pos, v in zip(huecos, resto):
            h[pos] = v
        return h
    return hijo(p1, p2), hijo(p2, p1)

def cruce_ciclos(p1, p2, cortes=None):
    pos_en_p1 = {v: i for i, v in enumerate(p1)}
    ciclo, i = set(), 0
    while i not in ciclo:                 # ciclo que empieza en la primera posición
        ciclo.add(i)
        i = pos_en_p1[p2[i]]
    h1 = [p1[i] if i in ciclo else p2[i] for i in range(len(p1))]
    h2 = [p2[i] if i in ciclo else p1[i] for i in range(len(p1))]
    return h1, h2

OPERADORES_CRUCE = {"monopunto": cruce_monopunto, "dos_puntos": cruce_dos_puntos,
                    "PMX": cruce_pmx, "OX": cruce_ox, "ciclos": cruce_ciclos}

# ---------------- MUTACIONES (devuelven un vector nuevo) ----------------
def mut_intercambio(genes, problema=None, posiciones=None):
    g = genes[:]
    i, j = posiciones if posiciones is not None else random.sample(range(len(g)), 2)
    g[i], g[j] = g[j], g[i]
    return g

def mut_insercion(genes, problema=None, posiciones=None):
    g = genes[:]
    i, j = posiciones if posiciones is not None else random.sample(range(len(g)), 2)
    v = g.pop(i)            # se saca el valor de la posición i ...
    g.insert(j, v)          # ... y se inserta en la posición j, desplazando el resto
    return g

def mut_inversion(genes, problema=None, posiciones=None):
    g = genes[:]
    a, b = posiciones if posiciones is not None else _dos_cortes(len(g))
    g[a:b] = g[a:b][::-1]
    return g

def mut_heuristica(genes, problema=None, posiciones=None, k=3):
    pos = posiciones if posiciones is not None else random.sample(range(len(genes)), min(k, len(genes)))
    mejor, mejor_f = None, None
    for perm in itertools.permutations([genes[p] for p in pos]):
        g = genes[:]
        for p, v in zip(pos, perm):
            g[p] = v
        f = problema["f"](g) if problema else contar_conflictos(g)
        objetivo = problema["objetivo"] if problema else "min"
        if mejor is None or es_mejor(f, mejor_f, objetivo):
            mejor, mejor_f = g, f
    return mejor

def mut_aleatoria(genes, problema=None, posiciones=None):
    g = genes[:]
    i = posiciones[0] if posiciones is not None else random.randrange(len(g))
    g[i] = random.randint(1, len(g))
    return g

OPERADORES_MUTACION = {"intercambio": mut_intercambio, "insercion": mut_insercion,
                       "inversion": mut_inversion, "heuristica": mut_heuristica,
                       "aleatoria": mut_aleatoria}

def mutacion(pob, prob_mut, tipo):
    operador = OPERADORES_MUTACION[tipo]
    for ind in pob:
        if random.random() < prob_mut:
            ind.genes = operador(ind.genes, ind.problema)
            ind.evaluar()

# ---------------- ALGORITMO GENÉTICO ----------------
def algoritmo_genetico(cfg=None, verbose=True):
    cfg = cfg or CONFIG
    if cfg["SEMILLA"] is not None and cfg["SEMILLA"] >= 0:
        random.seed(cfg["SEMILLA"])
        np.random.seed(cfg["SEMILLA"])
    problema = preparar_problema(cfg)
    objetivo = problema["objetivo"]
    tam_elite = max(1, round(cfg["PORC_ELITE"] * cfg["TAM_POB"])) if cfg["ELITISMO"] else 0
    operador = OPERADORES_CRUCE[cfg["CRUCE"]]
    cruce = lambda a, b: tuple(Individuo(problema, h) for h in operador(a.genes, b.genes))
    seleccion = SELECCIONES[cfg["SELECCION"]]
    inicio = time.time()

    pob = [Individuo(problema) for _ in range(cfg["TAM_POB"])]            # población inicial
    revisar_adaptacion(pob, objetivo)
    pos_mejor, _ = evaluacion(pob, objetivo)
    mejor_global = pob[pos_mejor].copia()
    historial = [registrar(pob, 0, pos_mejor)]
    soluciones = set(historial[0]["soluciones"])
    gen_solucion = 0 if soluciones else None
    eval_solucion = problema["evaluaciones"][0] if soluciones else None

    for g in range(1, cfg["NUM_GENERACIONES"] + 1):                      # bucle de evolución
        if cfg["DETENER_AL_ENCONTRAR"] and gen_solucion is not None:
            break                                                          # criterio de la sección 3.3
        elite = obtener_elite(pob, tam_elite, objetivo)
        pob = seleccion(pob, cfg)
        reproduccion(pob, cfg["PROB_CRUCE"], cruce)
        mutacion(pob, cfg["PROB_MUT"], cfg["MUTACION"])
        incluir_elite(pob, elite, objetivo)
        revisar_adaptacion(pob, objetivo)
        pos_mejor, _ = evaluacion(pob, objetivo)
        if es_mejor(pob[pos_mejor].adapt, mejor_global.adapt, objetivo):
            mejor_global = pob[pos_mejor].copia()
        h = registrar(pob, g, pos_mejor)
        historial.append(h)
        soluciones |= h["soluciones"]
        if gen_solucion is None and h["soluciones"]:
            gen_solucion, eval_solucion = g, h["evaluaciones"]

    res = dict(mejor=mejor_global, poblacion=pob, historial=historial, problema=problema,
               cfg=dict(cfg), tam_elite=tam_elite, soluciones=soluciones,
               gen_solucion=gen_solucion, eval_solucion=eval_solucion,
               evaluaciones=problema["evaluaciones"][0], tiempo=time.time() - inicio)
    if verbose:
        imprimir_resumen(res)
    return res

def imprimir_resumen(res):
    p, cfg, mejor = res["problema"], res["cfg"], res["mejor"]
    print("=" * 74)
    print(f"N = {p['N']} reinas  |  adaptación: {p['adaptacion']} ({p['formula']}, "
          f"{'minimizar' if p['objetivo'] == 'min' else 'maximizar'})")
    print(f"Selección: {cfg['SELECCION']}  |  cruce: {cfg['CRUCE']} ({cfg['PROB_CRUCE']})  |  "
          f"mutación: {cfg['MUTACION']} ({cfg['PROB_MUT']})  |  élite: {res['tam_elite']}")
    print("-" * 74)
    print(f"Mejor individuo : {mejor.cadena()}")
    print(f"Conflictos      : {mejor.conflictos}   →   {'ES SOLUCIÓN' if mejor.es_solucion() else 'no es solución'}")
    print(f"Adaptación      : {mejor.adapt:.6f}   (óptimo = {p['optimo']})")
    if res["gen_solucion"] is not None:
        print(f"Primera solución en la generación {res['gen_solucion']} "
              f"({res['eval_solucion']:,} evaluaciones de la función)")
    else:
        print("No se encontró ninguna solución en las generaciones indicadas.")
    total = SOLUCIONES_CONOCIDAS.get(p["N"])
    print(f"Soluciones distintas encontradas: {len(res['soluciones'])}"
          + (f" de {total:,} que existen" if total is not None else ""))
    print(f"Generaciones ejecutadas: {len(res['historial']) - 1}  |  "
          f"evaluaciones: {res['evaluaciones']:,}  |  tiempo: {res['tiempo']:.2f} s")
    print("=" * 74)

In [ ]:
#@title 7.1 Verificación de los operadores con los ejemplos del libro (sección 4.1.1)
P1 = [1, 3, 6, 4, 2, 5, 8, 7]
P2 = [2, 7, 8, 3, 1, 5, 4, 6]
P2_ciclos = [2, 7, 8, 1, 4, 5, 3, 6]
pruebas = [
    ("PMX, hijo 1 (pág. 98)",        cruce_pmx(P1, P2, (3, 6))[0],          [2, 4, 6, 3, 1, 5, 8, 7]),
    ("PMX, hijo 2 (pág. 98)",        cruce_pmx(P1, P2, (3, 6))[1],          [1, 7, 8, 4, 2, 5, 3, 6]),
    ("OX, hijo 1 (pág. 99)",         cruce_ox(P1, P2, (3, 6))[0],           [8, 3, 1, 4, 2, 5, 6, 7]),
    ("OX, hijo 2 (pág. 99)",         cruce_ox(P1, P2, (3, 6))[1],           [6, 4, 2, 3, 1, 5, 8, 7]),
    ("Ciclos, hijo 1 (pág. 100)",    cruce_ciclos(P1, P2_ciclos)[0],        [1, 7, 8, 4, 2, 5, 3, 6]),
    ("Ciclos, hijo 2 (pág. 100)",    cruce_ciclos(P1, P2_ciclos)[1],        [2, 3, 6, 1, 4, 5, 8, 7]),
    ("Inversión (pág. 103)",         mut_inversion(P1, posiciones=(2, 6)),  [1, 3, 5, 2, 4, 6, 8, 7]),
    ("Intercambio pos. 3 y 7 (pág. 103)", mut_intercambio(P1, posiciones=(2, 6)), [1, 3, 8, 4, 2, 5, 6, 7]),
    ("Inserción del 2 en 3.ª pos. (pág. 104)", mut_insercion(P1, posiciones=(4, 2)), [1, 3, 2, 6, 4, 5, 8, 7]),
]
tabla = pd.DataFrame([{"Operador": n, "Obtenido": str(o), "Libro": str(e), "¿Coincide?": "Sí" if o == e else "NO"}
                      for n, o, e in pruebas])
print("Todos los operadores coinciden con el libro." if (tabla["¿Coincide?"] == "Sí").all()
      else "ATENCIÓN: hay diferencias.")
tabla

## 8. Ejecución y resultados

In [ ]:
#@title 8. Ejecutar el algoritmo genético
res = algoritmo_genetico(CONFIG)

In [ ]:
#@title 8.1 Población inicial (tabla como las del libro)
def tabla_poblacion(res, generacion=0):
    h = res["historial"][generacion]
    return pd.DataFrame({"Posición": range(1, len(h["F"]) + 1),
                         "Individuo": ["[" + ", ".join(map(str, g)) + "]" for g in h["G"]],
                         "Conflictos": h["C"],
                         "Adaptación": np.round(h["F"], 6),
                         "¿Solución?": ["Sí" if c == 0 else "" for c in h["C"]]})

tabla_poblacion(res, generacion=0)

In [ ]:
#@title 8.2 Población final
tabla_poblacion(res, generacion=len(res["historial"]) - 1)

In [ ]:
#@title 8.3 Evolución por generación
def tabla_generaciones(res):
    return pd.DataFrame([{"Generación": h["generacion"],
                          "Adap. media": round(h["media"], 4),
                          "Mejor adap.": round(h["mejor_f"], 4),
                          "Conflictos del mejor": h["mejor_conf"],
                          "Conflictos (media)": round(h["media_conf"], 2),
                          "Soluciones en la población": int(sum(h["C"] == 0)),
                          "Individuos distintos": h["diversidad"],
                          "Mejor individuo": "[" + ", ".join(map(str, h["mejor_genes"])) + "]"}
                         for h in res["historial"]])

tabla_generaciones(res)

In [ ]:
#@title 8.4 Soluciones distintas encontradas
sols = sorted(res["soluciones"])
print(f"Se encontraron {len(sols)} soluciones distintas"
      + (f" de las {SOLUCIONES_CONOCIDAS[N]:,} que existen para N = {N}." if N in SOLUCIONES_CONOCIDAS else "."))
pd.DataFrame({"Solución": ["[" + ", ".join(map(str, s)) + "]" for s in sols]})

## 9. Gráficas: el tablero y cómo se mueven los individuos en cada generación
Funciones de graficación (se usan en la siguiente sección):
- `graficar_tableros` — el mejor individuo (tablero) en varias generaciones.
- `graficar_cada_generacion` — una gráfica por generación: tablero del mejor, ocupación de la población y distribución de conflictos.
- `animar_generaciones` — **animación con barra de reproducción** de lo anterior; opcionalmente guarda un GIF.
- `graficar_ocupacion` — mapa de calor: en qué casillas ponen sus reinas los individuos de la población en cada generación.
- `graficar_poblacion_genes` — la población completa como matriz (cada fila es un individuo, cada columna un gen).
- `graficar_dispersion` — la adaptación de cada individuo en cada generación.
- `graficar_convergencia` — mejor y media por generación, soluciones en la población y diversidad.
- `graficar_soluciones` — las soluciones distintas encontradas.

En los tableros, las reinas **negras** están a salvo y las **naranjas** están atacadas; las líneas naranjas unen a cada pareja que se ataca.

In [ ]:
#@title 9. Funciones de graficación
COLOR_MEJOR = "#0072B2"   # mejor individuo
COLOR_MEDIA = "#E69F00"   # media de la población
COLOR_SOL   = "#009E73"   # soluciones
COLOR_DIV   = "#CC79A7"   # diversidad
PALETA = ["#0072B2", "#E69F00", "#009E73", "#CC79A7", "#56B4E9", "#D55E00"]  # Okabe-Ito

def _gens_equiespaciadas(n_hist, paneles):
    return sorted(set(np.linspace(0, n_hist - 1, paneles).round().astype(int)))

def ocupacion(h, n):
    """Fracción de la población que tiene una reina en cada casilla (fila, columna)."""
    M = np.zeros((n, n))
    for g in h["G"]:
        M[g - 1, np.arange(n)] += 1
    return M / len(h["G"])

def _dibujar_ocupacion(ax, h, n, barra=None):
    im = ax.imshow(ocupacion(h, n), cmap="Blues", vmin=0, vmax=1,
                   extent=(0.5, n + 0.5, n + 0.5, 0.5))
    paso = 1 if n <= 12 else max(1, n // 8)
    ax.set_xticks(range(1, n + 1, paso)); ax.set_yticks(range(1, n + 1, paso))
    ax.xaxis.tick_top(); ax.tick_params(length=0, labelsize=8)
    if barra is not None:
        plt.colorbar(im, cax=barra, label="fracción de la población")
    return im

def _dibujar_histograma(ax, h, max_conf, tam_pob):
    conteo = np.bincount(h["C"], minlength=max_conf + 1)[:max_conf + 1]
    colores = [COLOR_SOL if c == 0 else COLOR_MEJOR for c in range(max_conf + 1)]
    ax.bar(range(max_conf + 1), conteo, color=colores, width=0.8)
    ax.set_xlim(-0.6, max_conf + 0.6)
    ax.xaxis.set_major_locator(plt.MaxNLocator(integer=True))
    ax.set_ylim(0, tam_pob)
    ax.set_xlabel("conflictos")
    ax.set_ylabel("individuos")
    ax.grid(True, axis="y", alpha=0.3)

def graficar_tableros(res, paneles=6, columnas=3):
    hist = res["historial"]
    gens = _gens_equiespaciadas(len(hist), paneles)
    filas = math.ceil(len(gens) / columnas)
    fig, axes = plt.subplots(filas, columnas, figsize=(4.0 * columnas, 4.4 * filas), squeeze=False,
                             constrained_layout=True)
    for ax, g in zip(axes.flat, gens):
        h = hist[g]
        dibujar_tablero(ax, h["mejor_genes"],
                        titulo=f"Generación {g} · {h['mejor_conf']} conflicto(s)"
                               + (" · SOLUCIÓN" if h["mejor_conf"] == 0 else ""))
    for ax in list(axes.flat)[len(gens):]:
        ax.axis("off")
    fig.suptitle(f"Mejor individuo por generación · N = {res['problema']['N']}", fontsize=12)
    plt.show()

def _figura_generacion(res):
    fig = plt.figure(figsize=(14, 4.6), dpi=80)
    gs = fig.add_gridspec(1, 5, width_ratios=[1, 1, 0.05, 0.32, 1.1], wspace=0.25,
                          left=0.03, right=0.98, top=0.80, bottom=0.13)
    return fig, fig.add_subplot(gs[0]), fig.add_subplot(gs[1]), fig.add_subplot(gs[2]), fig.add_subplot(gs[4])

def _pintar_generacion(res, g, axes, max_conf, con_barra=True):
    ax_t, ax_o, ax_b, ax_h = axes
    h = res["historial"][g]
    n = res["problema"]["N"]
    for ax in (ax_t, ax_o, ax_h):
        ax.clear()
    dibujar_tablero(ax_t, h["mejor_genes"],
                    titulo=f"Mejor individuo · {h['mejor_conf']} conflicto(s)")
    _dibujar_ocupacion(ax_o, h, n, barra=ax_b if con_barra else None)
    ax_o.set_title("Ocupación de la población", fontsize=9, pad=14)
    _dibujar_histograma(ax_h, h, max_conf, len(h["C"]))
    ax_h.set_title(f"Conflictos en la población · soluciones: {int(sum(h['C'] == 0))}", fontsize=9)
    ax_t.figure.suptitle(f"Generación {g}  ·  mejor f = {h['mejor_f']:.4f}  ·  media f = {h['media']:.4f}  ·  "
                         f"individuos distintos = {h['diversidad']}", fontsize=11)

def graficar_cada_generacion(res, cada=10, maximo=100):
    hist = res["historial"]
    gens = list(range(0, len(hist), max(1, cada)))
    if gens[-1] != len(hist) - 1:
        gens.append(len(hist) - 1)
    max_conf = int(max(h["C"].max() for h in hist))
    for g in gens[:maximo]:
        fig, *axes = _figura_generacion(res)
        _pintar_generacion(res, g, axes, max_conf)
        plt.show()

def animar_generaciones(res, paso=None, intervalo_ms=400, guardar_gif=False, nombre_gif="n_reinas.gif"):
    hist = res["historial"]
    if paso is None or paso <= 0:                       # automático: ~60 cuadros como máximo
        paso = max(1, math.ceil(len(hist) / 60))
    frames = list(range(0, len(hist), paso))
    if frames[-1] != len(hist) - 1:
        frames.append(len(hist) - 1)
    max_conf = int(max(h["C"].max() for h in hist))
    fig, *axes = _figura_generacion(res)
    _pintar_generacion(res, frames[0], axes, max_conf, con_barra=True)

    def actualizar(k):
        _pintar_generacion(res, frames[k], axes, max_conf, con_barra=False)
        return []

    anim = FuncAnimation(fig, actualizar, frames=len(frames), interval=intervalo_ms, blit=False)
    plt.close(fig)
    if guardar_gif:
        anim.save(nombre_gif, writer=PillowWriter(fps=max(1, round(1000 / intervalo_ms))))
        print(f"GIF guardado como '{nombre_gif}' (panel de archivos de la izquierda en Colab).")
    return HTML(anim.to_jshtml())

def graficar_ocupacion(res, paneles=6, columnas=3):
    hist, n = res["historial"], res["problema"]["N"]
    gens = _gens_equiespaciadas(len(hist), paneles)
    filas = math.ceil(len(gens) / columnas)
    fig, axes = plt.subplots(filas, columnas, figsize=(4.0 * columnas, 3.9 * filas), squeeze=False,
                             constrained_layout=True)
    for ax, g in zip(axes.flat, gens):
        im = _dibujar_ocupacion(ax, hist[g], n)
        ax.set_title(f"Generación {g}", fontsize=10, pad=14)
    for ax in list(axes.flat)[len(gens):]:
        ax.axis("off")
    fig.colorbar(im, ax=axes, shrink=0.8, label="fracción de la población con una reina en la casilla")
    fig.suptitle("¿Dónde ponen sus reinas los individuos? (fila × columna)", fontsize=12)
    plt.show()

def graficar_poblacion_genes(res, paneles=4):
    hist, n = res["historial"], res["problema"]["N"]
    gens = _gens_equiespaciadas(len(hist), paneles)
    objetivo = res["problema"]["objetivo"]
    fig, axes = plt.subplots(1, len(gens), figsize=(3.6 * len(gens), 5.4), squeeze=False,
                             constrained_layout=True)
    for ax, g in zip(axes[0], gens):
        h = hist[g]
        orden = np.argsort(h["F"])[::-1] if objetivo == "max" else np.argsort(h["F"])
        im = ax.imshow(h["G"][orden], aspect="auto", cmap="viridis", vmin=1, vmax=n,
                       interpolation="nearest", extent=(0.5, n + 0.5, len(orden) + 0.5, 0.5))
        ax.set_title(f"Generación {g}\n{h['diversidad']} individuos distintos", fontsize=10)
        ax.set_xlabel("columna (gen)")
        ax.set_xticks(range(1, n + 1, 1 if n <= 12 else max(1, n // 8)))
    axes[0, 0].set_ylabel("individuo (ordenados del mejor al peor)")
    fig.colorbar(im, ax=axes, shrink=0.85, label="fila de la reina (valor del gen)")
    fig.suptitle("La población como matriz de genes", fontsize=12)
    plt.show()

def graficar_dispersion(res):
    hist, p = res["historial"], res["problema"]
    fig, ax = plt.subplots(figsize=(10, 4.2))
    for h in hist:
        jitter = (np.random.rand(len(h["F"])) - 0.5) * 0.5
        ax.scatter(h["generacion"] + jitter, h["F"], s=7, color=COLOR_MEJOR, alpha=0.22, linewidths=0)
    ax.plot([h["generacion"] for h in hist], [h["mejor_f"] for h in hist], color=COLOR_MEDIA, lw=2,
            label="Mejor individuo")
    ax.axhline(p["optimo"], color="black", ls="--", lw=1, label=f"Óptimo ({p['optimo']:g})")
    ax.set_xlabel("Generación")
    ax.set_ylabel("adaptación f" + (" (conflictos)" if p["adaptacion"] == "crawford" else " (Homaifar)"))
    ax.set_title("Cada punto es un individuo: así se mueve la población generación tras generación", fontsize=10)
    ax.grid(True, alpha=0.3)
    ax.legend(loc="best")
    plt.tight_layout()
    plt.show()

def graficar_convergencia(res):
    hist, p = res["historial"], res["problema"]
    g = [h["generacion"] for h in hist]
    homaifar = p["adaptacion"] == "homaifar"
    filas = 3 if homaifar else 2
    fig, axes = plt.subplots(filas, 1, figsize=(12, 3.1 * filas), sharex=True, constrained_layout=True)
    ax = axes[0]
    ax.plot(g, [h["mejor_f"] for h in hist], color=COLOR_MEJOR, lw=2, label="Mejor de la generación")
    ax.plot(g, [h["media"] for h in hist], color=COLOR_MEDIA, lw=2, label="Media de la población")
    ax.axhline(p["optimo"], color="black", ls="--", lw=1, label=f"Óptimo ({p['optimo']:g})")
    ax.set_ylabel("adaptación f")
    ax.set_title(f"Evolución de la adaptación · {p['adaptacion']} · N = {p['N']}", fontsize=10)
    ax.legend(loc="upper left", bbox_to_anchor=(1.01, 1), fontsize=9); ax.grid(True, alpha=0.3)
    if homaifar:
        ax = axes[1]
        ax.plot(g, [h["mejor_conf"] for h in hist], color=COLOR_MEJOR, lw=2, label="Conflictos del mejor")
        ax.plot(g, [h["media_conf"] for h in hist], color=COLOR_MEDIA, lw=2, label="Conflictos (media)")
        ax.set_ylabel("conflictos"); ax.legend(loc="upper left", bbox_to_anchor=(1.01, 1), fontsize=9)
        ax.grid(True, alpha=0.3)
    ax = axes[-1]
    ax.plot(g, [int(sum(h["C"] == 0)) for h in hist], color=COLOR_SOL, lw=2, label="Soluciones en la población")
    ax.plot(g, [h["diversidad"] for h in hist], color=COLOR_DIV, lw=2, label="Individuos distintos")
    if res["gen_solucion"] is not None:
        ax.axvline(res["gen_solucion"], color="black", ls=":", lw=1,
                   label=f"Primera solución (gen. {res['gen_solucion']})")
    ax.set_ylabel("individuos"); ax.set_xlabel("Generación")
    ax.legend(loc="upper left", bbox_to_anchor=(1.01, 1), fontsize=9); ax.grid(True, alpha=0.3)
    plt.show()

def graficar_soluciones(res, maximo=12, columnas=4):
    sols = sorted(res["soluciones"])[:maximo]
    if not sols:
        print("No se encontraron soluciones para dibujar.")
        return
    filas = math.ceil(len(sols) / columnas)
    fig, axes = plt.subplots(filas, columnas, figsize=(3.2 * columnas, 3.8 * filas), squeeze=False,
                             constrained_layout=True)
    for ax, s in zip(axes.flat, sols):
        dibujar_tablero(ax, list(s))
    for ax in list(axes.flat)[len(sols):]:
        ax.axis("off")
    fig.suptitle(f"Soluciones distintas encontradas (se muestran {len(sols)} de {len(res['soluciones'])})", fontsize=12)
    plt.show()

## 10. Gráficas de los resultados

In [ ]:
#@title 10.1 El mejor individuo (tablero) en varias generaciones
PANELES = 6 #@param {type:"integer"}
graficar_tableros(res, paneles=PANELES)

In [ ]:
#@title 10.2 Animación de TODAS las generaciones (usa ▶ para reproducir)
PASO = 0 #@param {type:"integer"}
GUARDAR_GIF = False #@param {type:"boolean"}
# PASO = 0 elige el paso automáticamente (máximo ~60 cuadros); PASO = 1 muestra todas las generaciones.
display(animar_generaciones(res, paso=PASO, guardar_gif=GUARDAR_GIF, nombre_gif=f"n_reinas_N{N}.gif"))

In [ ]:
#@title 10.3 Una gráfica por generación
CADA = 25 #@param {type:"integer"}
# CADA = 1 dibuja absolutamente todas las generaciones (son muchas gráficas).
graficar_cada_generacion(res, cada=CADA)

In [ ]:
#@title 10.4 Ocupación del tablero por la población
graficar_ocupacion(res, paneles=6)

In [ ]:
#@title 10.5 La población como matriz de genes
graficar_poblacion_genes(res, paneles=4)

In [ ]:
#@title 10.6 Movimiento de los individuos (adaptación de cada uno por generación)
graficar_dispersion(res)

In [ ]:
#@title 10.7 Convergencia: mejor, media, soluciones y diversidad
graficar_convergencia(res)

In [ ]:
#@title 10.8 Soluciones distintas encontradas
graficar_soluciones(res, maximo=12)

## 11. Comparación con un algoritmo determinista (vuelta atrás o *backtracking*)
El libro aclara que las N reinas *"no es por naturaleza un problema de optimización, sino de búsqueda"*: una configuración es solución o no lo es. Por eso conviene compararlo con un algoritmo **determinista** clásico, la búsqueda con vuelta atrás: coloca una reina por columna, y si no hay fila segura regresa a la columna anterior.

- **Determinista:** con la misma $N$ siempre recorre los mismos pasos y encuentra las mismas soluciones, en el mismo orden (como en la definición vista en clase).
- **Algoritmo genético:** es probabilístico; cada semilla sigue un camino distinto y no garantiza encontrar todas las soluciones, pero escala a valores de $N$ en los que contar todas las soluciones es impracticable.

In [ ]:
#@title 11. Vuelta atrás: todas las soluciones y comparación con el AG
def backtracking(n, todas=True):
    soluciones, filas = [], [0] * n
    usadas, d1, d2 = set(), set(), set()
    def colocar(col):
        if col == n:
            soluciones.append(tuple(filas))
            return not todas                       # True = detenerse en la primera
        for fila in range(1, n + 1):
            if fila in usadas or (fila - col) in d1 or (fila + col) in d2:
                continue
            filas[col] = fila
            usadas.add(fila); d1.add(fila - col); d2.add(fila + col)
            if colocar(col + 1):
                return True
            usadas.discard(fila); d1.discard(fila - col); d2.discard(fila + col)
        return False
    colocar(0)
    return soluciones

N_MAX_BACKTRACKING = 10 #@param {type:"integer"}
filas_tabla = []
for n in range(4, N_MAX_BACKTRACKING + 1):
    t = time.time()
    s = backtracking(n)
    filas_tabla.append({"N": n, "Soluciones (vuelta atrás)": len(s),
                        "Valor conocido": SOLUCIONES_CONOCIDAS.get(n),
                        "Tiempo (s)": round(time.time() - t, 4), "Primera solución": str(list(s[0]))})
display(pd.DataFrame(filas_tabla))

if N <= N_MAX_BACKTRACKING:
    todas = set(backtracking(N))
    ga = res["soluciones"]
    print(f"\nN = {N}: el AG encontró {len(ga)} de las {len(todas)} soluciones "
          f"({100 * len(ga) / max(1, len(todas)):.1f} %). "
          f"Todas las del AG son válidas: {'Sí' if ga <= todas else 'NO'}")

## 12. Estudio de parámetros
Siguiendo al libro (*"fijamos todos los parámetros menos uno que vamos variando"*), `estudio_parametro(nombre, valores, repeticiones)` ejecuta el AG varias veces (semillas 0, 1, 2…) para cada valor, con el resto de parámetros como están en la celda 3. Para medir con justicia, en el estudio **se detiene al encontrar la primera solución**.

Con $N = 8$ casi cualquier configuración encuentra solución en pocas generaciones y las diferencias no se notan. Por eso el estudio usa su propio tablero, `N_ESTUDIO` (12 por defecto), y su propio límite, `GENERACIONES_ESTUDIO`; los demás parámetros se toman de la celda 3.

Columnas de la tabla:
- **éxito**: ejecuciones que encontraron al menos una solución.
- **generación promedio** y **evaluaciones promedio**: cuánto costó encontrar la primera solución (solo ejecuciones con éxito). Las evaluaciones son la medida más justa para comparar, porque no dependen del tamaño de la población.
- **conflictos finales**: promedio de conflictos del mejor al terminar (0 = siempre se resolvió).

La gráfica muestra los conflictos del mejor individuo por generación, promediados entre ejecuciones.

`comparar_opciones(nombre, opciones, repeticiones)` hace lo mismo con parámetros que no son numéricos: tipo de cruce, mutación, adaptación, selección o elitismo.

In [ ]:
#@title 12. Funciones del estudio de parámetros
N_ESTUDIO = 12 #@param {type:"integer"}
GENERACIONES_ESTUDIO = 200 #@param {type:"integer"}

def _cfg_estudio():
    cfg = dict(CONFIG)
    cfg.update(N=N_ESTUDIO, NUM_GENERACIONES=GENERACIONES_ESTUDIO)
    return cfg

def _ejecutar_varias(cambios, repeticiones):
    resultados = []
    for r in range(repeticiones):
        cfg = _cfg_estudio()
        cfg.update(cambios)
        cfg["SEMILLA"] = r
        cfg["DETENER_AL_ENCONTRAR"] = True
        resultados.append(algoritmo_genetico(cfg, verbose=False))
    return resultados

def _resumen(resultados, etiqueta, valor):
    exitos = [r for r in resultados if r["gen_solucion"] is not None]
    return {etiqueta: valor,
            "éxito": f"{len(exitos)}/{len(resultados)}",
            "% éxito": round(100 * len(exitos) / len(resultados), 1),
            "generación promedio": round(np.mean([r["gen_solucion"] for r in exitos]), 1) if exitos else "—",
            "evaluaciones promedio": int(np.mean([r["eval_solucion"] for r in exitos])) if exitos else "—",
            "conflictos finales": round(np.mean([r["mejor"].conflictos for r in resultados]), 2),
            "tiempo promedio (s)": round(np.mean([r["tiempo"] for r in resultados]), 3)}

def _curva(resultados, n_gen):
    curvas = []
    for r in resultados:
        c = [h["mejor_conf"] for h in r["historial"]]
        curvas.append(c + [c[-1]] * (n_gen + 1 - len(c)))   # tras encontrarla, sigue en 0
    return np.mean(curvas, axis=0)

def _graficar(curvas, titulo):
    fig, ax = plt.subplots(figsize=(10, 4.2))
    for k, (etiqueta, curva) in enumerate(curvas.items()):
        ax.plot(curva, lw=2, color=PALETA[k % len(PALETA)], label=str(etiqueta))
    ax.axhline(0, color="black", ls="--", lw=1, label="Solución (0 conflictos)")
    ax.set_xlabel("Generación")
    ax.set_ylabel("conflictos del mejor (promedio)")
    ax.set_title(titulo, fontsize=10)
    ax.grid(True, alpha=0.3)
    ax.legend(loc="best", fontsize=8)
    plt.tight_layout()
    plt.show()

def comparar_opciones(nombre, opciones, repeticiones=10):
    filas, curvas = [], {}
    for v in opciones:
        try:
            resultados = _ejecutar_varias({nombre: v}, repeticiones)
        except ValueError as e:
            filas.append({nombre: v, "éxito": "combinación no válida", "% éxito": None})
            print(f"{nombre} = {v}: {e}")
            continue
        filas.append(_resumen(resultados, nombre, v))
        n_gen = resultados[0]["cfg"]["NUM_GENERACIONES"]
        curvas[f"{nombre} = {v}"] = _curva(resultados, n_gen)
    n_txt = "varios" if nombre == "N" else N_ESTUDIO
    _graficar(curvas, f"{nombre} · N = {n_txt} · {repeticiones} ejecuciones por opción")
    return pd.DataFrame(filas)

def estudio_parametro(nombre, valores, repeticiones=10):
    return comparar_opciones(nombre, valores, repeticiones)

In [ ]:
#@title 12.1 Comparación de los cruces (sección 3.5.2.5 vs. sección 4.1.1.2)
comparar_opciones("CRUCE", ["monopunto", "dos_puntos", "PMX", "OX", "ciclos"], repeticiones=10)

In [ ]:
#@title 12.2 Comparación de las mutaciones
comparar_opciones("MUTACION", ["intercambio", "insercion", "inversion", "heuristica"], repeticiones=10)

In [ ]:
#@title 12.3 Crawford (penalización) vs. Homaifar (premio)
comparar_opciones("ADAPTACION", ["crawford", "homaifar"], repeticiones=10)

In [ ]:
#@title 12.4 Ruleta vs. torneo
comparar_opciones("SELECCION", ["ruleta", "torneo"], repeticiones=10)

In [ ]:
#@title 12.5 Con vs. sin elitismo
comparar_opciones("ELITISMO", [False, True], repeticiones=10)

In [ ]:
#@title 12.6 Porcentaje de cruces (el libro lo recomienda reducido con monopunto o dos puntos)
CRUCE_A_ESTUDIAR = "monopunto" #@param ["monopunto", "dos_puntos", "PMX", "OX", "ciclos"]
_cruce_original = CONFIG["CRUCE"]
CONFIG["CRUCE"] = CRUCE_A_ESTUDIAR
try:
    display(estudio_parametro("PROB_CRUCE", [0.0, 0.2, 0.4, 0.6, 0.8, 1.0], repeticiones=10))
finally:
    CONFIG["CRUCE"] = _cruce_original

In [ ]:
#@title 12.7 Probabilidad de mutación
estudio_parametro("PROB_MUT", [0.0, 0.05, 0.1, 0.2, 0.4, 0.8], repeticiones=10)

In [ ]:
#@title 12.8 Tamaño de la población
estudio_parametro("TAM_POB", [20, 50, 100, 200], repeticiones=10)

In [ ]:
#@title 12.9 Tamaño del tablero N
estudio_parametro("N", [4, 8, 12, 16, 20, 25], repeticiones=5)

## 13. Cómo agregar lo que pida el doctor
- **Otra función de adaptación:** escríbela en la celda 2 (recibe la lista de genes) y agrégala a `preparar_problema` (celda 5) con su objetivo (`"min"` o `"max"`) y su óptimo; después añade su nombre a la lista `ADAPTACION` del formulario de la celda 3.
- **Otro cruce:** escribe `cruce_nuevo(p1, p2)` en la celda 7 (recibe y devuelve listas de genes), agrégalo a `OPERADORES_CRUCE` y a la lista `CRUCE` del formulario. Si conserva las permutaciones, agrégalo también a `CRUCES_PERMUTACION` (celda 2).
- **Otra mutación:** escribe `mut_nueva(genes, problema=None, posiciones=None)` en la celda 7, agrégala a `OPERADORES_MUTACION` y a la lista `MUTACION` del formulario.
- **Otra selección:** escribe `seleccion_nueva(pob, cfg)` en la celda 6 y agrégala a `SELECCIONES` y al formulario.
- **Penalización variable** (sección 3.5.1, la penalización crece conforme avanza la evolución): pasa el número de generación a la función de adaptación dentro del bucle de `algoritmo_genetico`.